# Redes e a internet

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Acompanhar um pedido da web pelo DNS, TCP e HTTP, e escolher TCP ou UDP para uma aplicação
- Verificar se um endereço IPv4 é válido e lê-lo como um único número de 32 bits
- Achar os endereços de rede, de broadcast e de hosts de um endereço com prefixo
- Escolher o próximo salto de um pacote pelo maior prefixo correspondente, como um roteador faz

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

A internet nunca manda um arquivo de uma vez. Ela corta os dados em **pacotes** de cerca de 1500 bytes, cada um com um **cabeçalho** (de onde vem e para onde vai) e um pedaço dos dados. Os **roteadores** passam cada pacote um salto mais perto do destino; os pacotes podem seguir caminhos diferentes e chegar fora de ordem.

**Cortando um arquivo em pacotes:**

In [ ]:
int dados = 1460; // bytes de dados num pacote típico de 1500 bytes, depois de 40 bytes de cabeçalhos
for (long tamanho : new long[] {1_000, 100_000, 5_000_000}) {
    long pacotes = (tamanho + dados - 1) / dados; // arredondado para cima
    IO.println(String.format("%,10d bytes: %,6d pacotes", tamanho, pacotes));
}

Todo dispositivo tem um **endereço IP**: 4 bytes escritos como `192.168.1.130`, que para o computador são um único número de 32 bits. Um **prefixo** como `/26` diz quantos desses bits identificam a **rede**; o resto numera os **hosts** dentro dela.

### ✏️ Exercício: É um octeto?

Cada parte de um endereço IPv4 é um número de 0 a 255, escrito com 1 a 3 dígitos. Confira uma parte.

In [ ]:
boolean isOctet(String part) {
    // TODO: confira o tamanho, se todo char é dígito, e o intervalo
    return false;
}

void main() {
    check("192", isOctet("192"), true);
    check("256", isOctet("256"), false);
    check("1a", isOctet("1a"), false);
    check("vazio", isOctet(""), false);
}

<details>
<summary>Ver uma resposta</summary>

```java
boolean isOctet(String part) {
    if (part.isEmpty() || part.length() > 3) {
        return false;
    }
    for (char c : part.toCharArray()) {
        if (c < '0' || c > '9') {
            return false;
        }
    }
    return Integer.parseInt(part) <= 255;
}

void main() {
    check("192", isOctet("192"), true);
    check("256", isOctet("256"), false);
    check("1a", isOctet("1a"), false);
    check("vazio", isOctet(""), false);
}
```

</details>

![O endereço 192.168.1.130/26 dividido em bits de rede e de host](prefix.pt-br.svg)

#### 🎞️ Dividindo um endereço com um prefixo

![Para 192.168.1.130/26 os dois primeiros bits do último byte nomeiam a rede: rede .128, broadcast .191, dispositivos .129 a .190](subnet-steps.pt-br.svg)

Todos os bits de host 0 é o **endereço de rede**, todos 1 é o **endereço de broadcast**, e os endereços do meio são para os dispositivos. Um roteador escolhe para onde mandar um pacote pelo **maior prefixo** da sua tabela que corresponde ao destino.

**Quantos endereços cada prefixo dá?**

In [ ]:
for (int prefixo = 22; prefixo <= 30; prefixo++) {
    long enderecos = 1L << (32 - prefixo);
    IO.println(String.format("/%d: %,5d endereços, %,5d para dispositivos", prefixo, enderecos, enderecos - 2));
}

### ✏️ Exercício: Dispositivos numa rede

Um prefixo /p deixa 32 − p bits para os hosts, então 2³²⁻ᵖ endereços. Dois deles (o de rede e o de broadcast) não podem ser usados por dispositivos.

In [ ]:
long devices(int prefix) {
    // TODO: conte os endereços com um deslocamento, menos 2
    return 0;
}

void main() {
    check("/24", devices(24), 254L);
    check("/26", devices(26), 62L);
    check("/30", devices(30), 2L);
}

<details>
<summary>Ver uma resposta</summary>

```java
long devices(int prefix) {
    return (1L << (32 - prefix)) - 2;
}

void main() {
    check("/24", devices(24), 254L);
    check("/26", devices(26), 62L);
    check("/30", devices(30), 2L);
}
```

</details>

Por cima do IP, o **TCP** numera os dados, confirma o que chegou e reenvia o que se perdeu, para que páginas e arquivos cheguem completos e em ordem. O **UDP** pula tudo isso, o que serve para chamadas de vídeo e jogos, em que um pacote atrasado não serve para nada. Antes de conectar, seu computador pergunta ao **DNS** o endereço de um nome como `example.com`. Depois o navegador abre uma conexão TCP e manda um pedido **HTTP**: `GET /index.html HTTP/1.1`.

## Leituras

- [The Internet](https://www.youtube.com/watch?v=AEaKrq3SpW8) (Crash Course Computer Science #29, 12 min): Como uma mensagem vira pacotes, como os roteadores os passam adiante salto a salto, por que o TCP numera e confirma os pacotes e o UDP não, e o que o DNS faz antes de tudo isso.
- [How DNS Works](https://howdns.works/) (DNSimple (a comic in 6 short episodes), 15 min): Acompanhe um nome desde o cache do navegador até o resolvedor, depois os servidores raiz, de TLD e autoritativo, e repare onde cada resposta fica em cache para a próxima vez.
- [Subnet Cheat Sheet](https://www.freecodecamp.org/news/subnet-cheat-sheet-24-subnet-mask-30-26-27-29-and-other-ip-address-cidr-network-references/) (freeCodeCamp, 9 min): Como um prefixo como /26 divide um endereço em bits de rede e de host, como uma máscara em binário marca a parte de rede, e a tabela de prefixos com os endereços utilizáveis (/26 tem 62). As máscaras curinga e o caso especial do /31 são mais do que você precisa.

## Verifique

1. Por que uma chamada de vídeo usa UDP enquanto um download usa TCP?
2. Quais são os endereços de rede e de broadcast de `10.20.30.40/24`?
3. Um roteador tem rotas para `10.0.0.0/8` e `10.1.0.0/16`. Para onde ele manda um pacote para `10.1.2.3`, e por quê?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Endereço IPv4 válido** · Fácil
- **Lendo um pedido HTTP** · Fácil
- **Calculadora de sub-rede** · Médio
- **Tabela de roteamento** · Difícil
- Teste: **Inspetor de pacotes** · Difícil